# Binance Spot hourly klines for the initial USDT universe

This notebook collects closed `1h` Binance Spot klines for the 30-pair initial universe approved in `docs/methodology_decisions.md`. The requested period is fixed from `2017-01-01 00:00:00 UTC` through `2026-09-01 23:59:59.999 UTC`. Each pair starts at its first available Binance candle; observations are not synthesized before listing or across missing intervals.

For each pair, the notebook creates a dedicated directory under `data/raw/binance/spot/1h/`. The canonical raw artifact is a `.csv.gz` file that preserves Binance decimal fields as text. A derived Excel review copy contains `Klines`, `Metadata`, `Validation`, and `Missing Intervals` sheets, and a JSON sidecar records checksums and validation results. A timestamped collection manifest summarizes the execution. Final artifacts are validated and never overwritten.

## Research limitation

The universe was selected ex post using activity observed through 2026-09-01 and current symbol status. These data may support exploration and later protocol development, but the fixed universe must not be presented as historically known at earlier portfolio decision timestamps.

In [5]:
import importlib
from pathlib import Path
import sys

import pandas as pd


def find_repository_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or project directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import tcc_crypto.binance_hourly_collection as collection_module

collection_module = importlib.reload(collection_module)
load_collection_config = collection_module.load_collection_config
run_collection = collection_module.run_collection

CONFIG_PATH = (
    REPOSITORY_ROOT
    / "configs"
    / "binance_selected_usdt_hourly_collection.json"
)
config = load_collection_config(CONFIG_PATH)
print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Configuration: {CONFIG_PATH}")

Repository root: /Users/marcelapsilverio/projetos/tcc_crypto
Configuration: /Users/marcelapsilverio/projetos/tcc_crypto/configs/binance_selected_usdt_hourly_collection.json


In [6]:
configuration_summary = pd.DataFrame(
    [
        ("Collection ID", config.collection_id),
        ("Market", config.market),
        ("Interval", config.interval),
        ("Requested start (UTC)", config.start_time_utc.isoformat()),
        ("Requested end (UTC)", config.end_time_utc.isoformat()),
        ("Selected pairs", len(config.symbols)),
        ("Output directory", str(REPOSITORY_ROOT / config.output_directory)),
        ("Source endpoint", f"{config.base_url}{config.kline_path}"),
    ],
    columns=["Field", "Value"],
)
display(configuration_summary)
display(
    pd.DataFrame(
        {
            "Selection Rank": range(1, len(config.symbols) + 1),
            "Trading Pair": config.symbols,
        }
    )
)

,Field,Value
0,Collection ID,binance_selected_usdt_hourly_2017_2026-09-01
1,Market,Binance Spot
2,Interval,1h
3,Requested start (UTC),2017-01-01T00:00:00+00:00
4,Requested end (UTC),2026-09-01T23:59:59.999000+00:00
5,Selected pairs,30
6,Output directory,/Users/marcelapsilverio/projetos/tcc_crypto/da...
7,Source endpoint,https://data-api.binance.vision/api/v3/klines


,Selection Rank,Trading Pair
0,1,BTCUSDT
1,2,FLOKIUSDT
2,3,ETHUSDT
3,4,BONKUSDT
4,5,SOLUSDT
5,6,XRPUSDT
6,7,SHIBUSDT
7,8,DOGEUSDT
8,9,BNBUSDT
9,10,ADAUSDT


## Run the collection

The following cell performs the network requests and writes the pair artifacts. It can be rerun after an interruption: collection resumes after the last page persisted in the pair checkpoint. Complete existing CSV and workbook artifacts are validated and reported as `VALIDATED_EXISTING`; their counts, dates, gaps, paths, and checksums remain populated in the new manifest. Failed pairs are recorded without deleting successful outputs.

In [7]:
collection_results, manifest_path = run_collection(
    config,
    REPOSITORY_ROOT,
    progress_callback=print,
)
display(collection_results)
print(f"Collection manifest: {manifest_path}")

Completed 1/30: FLOKIUSDT (COLLECTED)
Completed 2/30: BTCUSDT (COLLECTED)
Completed 3/30: BONKUSDT (COLLECTED)
Completed 4/30: ETHUSDT (COLLECTED)
Completed 5/30: XRPUSDT (FAILED)
Completed 6/30: SOLUSDT (FAILED)
Completed 7/30: DOGEUSDT (FAILED)
Completed 8/30: SHIBUSDT (FAILED)
Completed 9/30: ADAUSDT (FAILED)
Completed 10/30: BNBUSDT (FAILED)
Completed 11/30: PEPEUSDT (COLLECTED)
Completed 12/30: SUIUSDT (COLLECTED)
Completed 13/30: LTCUSDT (COLLECTED)
Completed 14/30: AVAXUSDT (COLLECTED)
Completed 15/30: LINKUSDT (COLLECTED)
Completed 16/30: DOTUSDT (COLLECTED)
Completed 17/30: TRXUSDT (COLLECTED)
Completed 18/30: WIFUSDT (COLLECTED)
Completed 19/30: NEIROUSDT (COLLECTED)
Completed 20/30: NEARUSDT (COLLECTED)
Completed 21/30: AAVEUSDT (COLLECTED)
Completed 22/30: WLDUSDT (COLLECTED)
Completed 23/30: XLMUSDT (COLLECTED)
Completed 24/30: ZECUSDT (COLLECTED)
Completed 25/30: FILUSDT (COLLECTED)
Completed 26/30: BCHUSDT (COLLECTED)
Completed 27/30: LUNAUSDT (COLLECTED)
Completed 28/30

,Selection Rank,Trading Pair,Status,Rows,Missing Hours,Shortened Candles,First Open (UTC),Last Close (UTC),Raw CSV,Raw CSV SHA-256,Workbook,Workbook SHA-256,Metadata,Error
0,1,BTCUSDT,COLLECTED,79140.0,128.0,14.0,2017-08-17T04:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,34fe0dc397a752ce7be95825d3caf0126f32a0aaf13dea...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,3973e833f0b1f2d5ddab5008a9b3b08a94bd9adcb799b6...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,NaN
1,2,FLOKIUSDT,COLLECTED,29166.0,0.0,0.0,2023-05-05T18:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,1a6334deffa46f0c13ff288b05932d35b540bfe9176112...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,0edf39780972b6e6a6e03dafc55dbf1359a00fe86ccb60...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,NaN
2,3,ETHUSDT,COLLECTED,79140.0,128.0,14.0,2017-08-17T04:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,9e58f8911779f297397da6d71e71f2e85a328322e7a690...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,31aac275eea4fe1e85054684ca48c518c450a9b0eb1825...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,NaN
3,4,BONKUSDT,COLLECTED,23800.0,0.0,0.0,2023-12-15T08:00:00+00:00,2026-09-01T23:59:59.999000+00:00,/Users/marcelapsilverio/projetos/tcc_crypto/da...,0ec9b70af0d1f08a6c80a1f1582fd89d8bed450c276e23...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,22b6ffa98e25e7788aada1d8e59cc3f6a89bca76614382...,/Users/marcelapsilverio/projetos/tcc_crypto/da...,NaN
4,5,SOLUSDT,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remote end closed connection without response
5,6,XRPUSDT,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"IncompleteRead(97568 bytes read, 72053 more ex..."
6,7,SHIBUSDT,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"IncompleteRead(85782 bytes read, 79321 more ex..."
7,8,DOGEUSDT,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"IncompleteRead(33291 bytes read, 124865 more e..."
8,9,BNBUSDT,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remote end closed connection without response
9,10,ADAUSDT,FAILED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remote end closed connection without response


Collection manifest: /Users/marcelapsilverio/projetos/tcc_crypto/data/raw/binance/spot/1h/collection_manifest_20260903T024652412977Z.xlsx


In [8]:
status_summary = (
    collection_results["Status"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="Pairs")
)
display(status_summary)

failed = collection_results.loc[collection_results["Status"] == "FAILED"]
pairs_requiring_gap_review = collection_results.loc[
    collection_results["Missing Hours"].fillna(0) > 0
]
if not pairs_requiring_gap_review.empty:
    print("Pairs with missing hourly intervals requiring review:")
    display(
        pairs_requiring_gap_review[
            ["Trading Pair", "Missing Hours", "Workbook"]
        ]
    )

pairs_with_shortened_candles = collection_results.loc[
    collection_results["Shortened Candles"].fillna(0) > 0
]
if not pairs_with_shortened_candles.empty:
    print("Pairs with shortened historical candles requiring review:")
    display(
        pairs_with_shortened_candles[
            ["Trading Pair", "Shortened Candles", "Workbook"]
        ]
    )

if not failed.empty:
    display(failed[["Trading Pair", "Error"]])
    raise RuntimeError(
        f"Collection finished with {len(failed)} failed pair(s). "
        "Review the manifest and rerun the collection cell."
    )

print("All configured pairs were collected or already existed.")

,Status,Pairs
0,COLLECTED,24
1,FAILED,6


Pairs with missing hourly intervals requiring review:


,Trading Pair,Missing Hours,Workbook
0,BTCUSDT,128.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
2,ETHUSDT,128.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
10,LTCUSDT,122.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
13,LINKUSDT,60.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
14,AVAXUSDT,20.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
15,DOTUSDT,20.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
16,TRXUSDT,88.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
18,NEARUSDT,20.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
20,AAVEUSDT,20.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
22,XLMUSDT,88.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...


Pairs with shortened historical candles requiring review:


,Trading Pair,Shortened Candles,Workbook
0,BTCUSDT,14.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
2,ETHUSDT,14.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
10,LTCUSDT,13.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
13,LINKUSDT,9.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
14,AVAXUSDT,6.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
15,DOTUSDT,6.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
16,TRXUSDT,9.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
18,NEARUSDT,6.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
20,AAVEUSDT,6.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...
22,XLMUSDT,9.0,/Users/marcelapsilverio/projetos/tcc_crypto/da...


,Trading Pair,Error
4,SOLUSDT,Remote end closed connection without response
5,XRPUSDT,"IncompleteRead(97568 bytes read, 72053 more ex..."
6,SHIBUSDT,"IncompleteRead(85782 bytes read, 79321 more ex..."
7,DOGEUSDT,"IncompleteRead(33291 bytes read, 124865 more e..."
8,BNBUSDT,Remote end closed connection without response
9,ADAUSDT,Remote end closed connection without response


RuntimeError: Collection finished with 6 failed pair(s). Review the manifest and rerun the collection cell.

## Interpretation and next checks

A missing interval inside the observed span is recorded as `REVIEW`; it is not filled in the canonical raw CSV. A candle whose close remains inside its nominal hour but occurs before the final millisecond is also preserved and marked `REVIEW`, with details in `Nonstandard Durations`. A close before the open or beyond the nominal hour remains a fatal validation failure. Time before a pair's first observed candle is treated as pre-listing history rather than missing data. The Excel workbooks are derived inspection artifacts and must not replace the CSV files as pipeline inputs.